In [30]:
# Load the Netflix dataset from the attached Kaggle dataset
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv


In [31]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv


In [32]:
import pandas as pd

filepath = "/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv"
df = pd.read_csv(filepath)

df.info()
print("Shape of the dataset (R x C):", df.shape)
print("Missing values per column:\n", df.isnull().sum())
print("Number of duplicate rows:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB
Shape of the dataset (R x C): (8807, 12)
Missing values per column:
 show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating         

In [33]:
def clean_column_names(d):
    """Strips whitespace and lowercases column names, replacing spaces with underscores."""
    d.columns = d.columns.str.strip().str.lower().str.replace(" ", "_")
    return d

df = clean_column_names(df)

In [34]:
df["date_added"] = df["date_added"].str.strip()
df["date_added"] = pd.to_datetime(df["date_added"], format="mixed")

dur = df["duration"].str.extract(r"(\d+)\s*(\w+)")
df["duration_value"] = pd.to_numeric(dur[0])
df["duration_unit"] = dur[1]

print(df[["date_added", "duration", "duration_value", "duration_unit"]].head())

  date_added   duration  duration_value duration_unit
0 2021-09-25     90 min            90.0           min
1 2021-09-24  2 Seasons             2.0       Seasons
2 2021-09-24   1 Season             1.0        Season
3 2021-09-24   1 Season             1.0        Season
4 2021-09-24  2 Seasons             2.0       Seasons


In [35]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} exact duplicate rows.")

df = df.drop(columns=["description"])

Dropped 0 exact duplicate rows.


In [36]:
# Build director/cast pair lookup from pairs that repeat 3+ times with a
# real director attached.
work = df.dropna(subset=["cast"]).copy()
work["dir_cast"] = work["director"].fillna("") + "---" + work["cast"]
counts = work["dir_cast"].value_counts()
filtered = counts[counts >= 3]

dict_direcast = {}
for combo in filtered.index:
    director, cast = combo.split("---", 1)
    if director:
        dict_direcast[cast] = director

filled_directors = 0
for cast_val, director_val in dict_direcast.items():
    mask = df["director"].isna() & (df["cast"] == cast_val)
    filled_directors += int(mask.sum())
    df.loc[mask, "director"] = director_val
print(f"Filled {filled_directors} missing directors from repeated director/cast pairs.")

df.loc[df["director"].isna(), "director"] = "Not Given"

# Director -> country lookup from rows that already have a country.
dir_country = (
    df.dropna(subset=["country"])
    .drop_duplicates(subset=["director"])
    .set_index("director")["country"]
    .to_dict()
)
filled_countries = 0
for director_val, country_val in dir_country.items():
    mask = df["country"].isna() & (df["director"] == director_val)
    filled_countries += int(mask.sum())
    df.loc[mask, "country"] = country_val
print(f"Filled {filled_countries} missing countries from director lookup.")

df.loc[df["country"].isna(), "country"] = "Not Given"
df.loc[df["cast"].isna(), "cast"] = "Not Given"

before = len(df)
df = df.dropna(subset=["date_added", "rating", "duration"])
print(f"Dropped {before - len(df)} rows with missing date_added/rating/duration.")

print(df.isnull().sum())

Filled 18 missing directors from repeated director/cast pairs.
Filled 557 missing countries from director lookup.
Dropped 17 rows with missing date_added/rating/duration.
show_id           0
type              0
title             0
director          0
cast              0
country           0
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
duration_value    0
duration_unit     0
dtype: int64


In [37]:
mismatched_mask = df["date_added"].dt.year < df["release_year"]
print(f"Rows where date_added year < release_year: {mismatched_mask.sum()}")
print(df.loc[mismatched_mask, ["title", "release_year", "date_added"]].head(10))

df = df.loc[~mismatched_mask].copy()
print(f"Dropped {mismatched_mask.sum()} rows with an impossible add date.")

Rows where date_added year < release_year: 14
                           title  release_year date_added
1551                       Hilda          2021 2020-12-14
1696                Polly Pocket          2021 2020-11-15
2920               Love Is Blind          2021 2020-02-13
3168                Fuller House          2020 2019-12-06
3287          Maradona in Mexico          2020 2019-11-13
3369             BoJack Horseman          2020 2019-10-25
3433            The Hook Up Plan          2020 2019-10-11
4844   Unbreakable Kimmy Schmidt          2019 2018-05-30
4845        Arrested Development          2019 2018-05-29
5394  Hans Teeuwen: Real Rancour          2018 2017-07-01
Dropped 14 rows with an impossible add date.


In [38]:
pre_1997_mask = df["date_added"].dt.year < 1997
print(f"Rows where date_added year < 1997: {pre_1997_mask.sum()}")
print(df.loc[pre_1997_mask, ["title", "date_added"]].head(10))

df = df.loc[~pre_1997_mask].copy()
print(f"Dropped {pre_1997_mask.sum()} rows added before Netflix existed.")

Rows where date_added year < 1997: 0
Empty DataFrame
Columns: [title, date_added]
Index: []
Dropped 0 rows added before Netflix existed.


In [39]:
print("type unique:", df["type"].unique())
print("rating unique:", df["rating"].unique())


type unique: ['Movie' 'TV Show']
rating unique: ['PG-13' 'TV-MA' 'PG' 'TV-14' 'TV-PG' 'TV-Y' 'TV-Y7' 'R' 'TV-G' 'G'
 'NC-17' 'NR' 'TV-Y7-FV' 'UR']


In [40]:
print(df.loc[mismatched_mask, ["title", "release_year", "date_added"]].head(10))

Empty DataFrame
Columns: [title, release_year, date_added]
Index: []


In [41]:
split_cols = df["listed_in"].str.split(r",\s*", expand=True)
max_cats = split_cols.shape[1]
print("Max categories found:", max_cats)

split_cols.columns = [f"listed_in_{i+1}" for i in range(max_cats)]
df = pd.concat([df.drop(columns=["listed_in"]), split_cols], axis=1)

print(df[[c for c in df.columns if c.startswith("listed_in")]].head())

Max categories found: 3
              listed_in_1             listed_in_2            listed_in_3
0           Documentaries                    None                   None
1  International TV Shows               TV Dramas           TV Mysteries
2          Crime TV Shows  International TV Shows  TV Action & Adventure
3              Docuseries              Reality TV                   None
4  International TV Shows       Romantic TV Shows            TV Comedies


In [42]:
assert pd.api.types.is_datetime64_any_dtype(df["date_added"]), "date_added is not datetime!"
assert pd.api.types.is_numeric_dtype(df["duration_value"]), "duration_value is not numeric!"
print("dtype checks passed.")

assert df["director"].isnull().sum() == 0, "director still has missing values!"
assert df["country"].isnull().sum() == 0, "country still has missing values!"
print("no missing director/country left.")

df = df.reset_index(drop=True)
df.sample(5)

dtype checks passed.
no missing director/country left.


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,duration_value,duration_unit,listed_in_1,listed_in_2,listed_in_3
4579,s4587,Movie,Mugamoodi,Mysskin,"Jiiva, Pooja Hegde, Narain, Nasser, Girish Kar...",India,2018-10-01,2012,TV-14,150 min,150.0,min,Action & Adventure,International Movies,None
1170,s1171,Movie,Secret Magic Control Agency,Aleksey Tsitsilin,"Nicholas Corda, Sylvana Joyce, Alyson Leigh Ro...","Russia, United States",2021-03-25,2021,TV-Y7,105 min,105.0,min,Children & Family Movies,Comedies,None
8476,s8508,Movie,The Signal,"William Eubank, Will Eubank","Brenton Thwaites, Olivia Cooke, Laurence Fishb...",United States,2018-07-13,2014,PG-13,97 min,97.0,min,Sci-Fi & Fantasy,Thrillers,None
6886,s6909,Movie,Hadi İnşallah,Ali Taner Baltacı,"Büşra Pekin, Murat Boz, Cezmi Baskın, Şinasi Y...",Turkey,2018-04-04,2014,TV-14,95 min,95.0,min,Comedies,International Movies,Romantic Movies
691,s692,TV Show,So Not Worth It,Not Given,"Park Se-wan, Shin Hyeon-seung, Choi Young-jae,...",South Africa,2021-06-18,2021,TV-14,1 Season,1.0,Season,International TV Shows,TV Comedies,None


In [43]:
output_filename = "cleaned_netflix_.Karen_Kinyua.csv"   # replace 'yourname' with your actual name
df.to_csv(output_filename, index=False)

check_df = pd.read_csv(output_filename)
print(f"Saved and reloaded {check_df.shape[0]} rows, {check_df.shape[1]} columns.")
check_df.head()

Saved and reloaded 8776 rows, 15 columns.


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,duration_value,duration_unit,listed_in_1,listed_in_2,listed_in_3
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Not Given,United States,2021-09-25,2020,PG-13,90 min,90.0,min,Documentaries,NaN,NaN
1,s2,TV Show,Blood & Water,Not Given,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,2.0,Seasons,International TV Shows,TV Dramas,TV Mysteries
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",France,2021-09-24,2021,TV-MA,1 Season,1.0,Season,Crime TV Shows,International TV Shows,TV Action & Adventure
3,s4,TV Show,Jailbirds New Orleans,Not Given,Not Given,South Africa,2021-09-24,2021,TV-MA,1 Season,1.0,Season,Docuseries,Reality TV,NaN
4,s5,TV Show,Kota Factory,Not Given,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,2.0,Seasons,International TV Shows,Romantic TV Shows,TV Comedies


In [44]:
check_df = pd.read_csv(output_filename)
print(f"Saved and reloaded {check_df.shape[0]} rows, {check_df.shape[1]} columns.")
check_df.head()

Saved and reloaded 8776 rows, 15 columns.


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,duration_value,duration_unit,listed_in_1,listed_in_2,listed_in_3
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Not Given,United States,2021-09-25,2020,PG-13,90 min,90.0,min,Documentaries,NaN,NaN
1,s2,TV Show,Blood & Water,Not Given,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,2.0,Seasons,International TV Shows,TV Dramas,TV Mysteries
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",France,2021-09-24,2021,TV-MA,1 Season,1.0,Season,Crime TV Shows,International TV Shows,TV Action & Adventure
3,s4,TV Show,Jailbirds New Orleans,Not Given,Not Given,South Africa,2021-09-24,2021,TV-MA,1 Season,1.0,Season,Docuseries,Reality TV,NaN
4,s5,TV Show,Kota Factory,Not Given,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,2.0,Seasons,International TV Shows,Romantic TV Shows,TV Comedies
